In [2]:
import os

def print_tree(path, indent=""):
    abs_path = os.path.abspath(path)
    print(indent + abs_path + "/")   # 打印绝对路径
    indent += "    "
    for item in sorted(os.listdir(path)):
        full_path = os.path.join(path, item)
        if os.path.isdir(full_path):
            print_tree(full_path, indent)  # 递归子文件夹
        else:
            print(indent + os.path.abspath(full_path))  # 文件绝对路径

# 使用方法
# generalizability数据集
root = "datasets"
print_tree(root)

# expanded数据集
root = "../1D_Burgers/datasets/1D/Burgers/expanded_pos/t1"
print_tree(root)

# train test 数据集
root = "test_train_datasets"
print_tree(root)

/blue/shiboli.fsu/yifeisun.umich/adversarial_robustness_FNO/1D_Burgers_FNO_generalization/datasets/
    /blue/shiboli.fsu/yifeisun.umich/adversarial_robustness_FNO/1D_Burgers_FNO_generalization/datasets/all_neg/
        /blue/shiboli.fsu/yifeisun.umich/adversarial_robustness_FNO/1D_Burgers_FNO_generalization/datasets/all_neg/gaussian_diff_distr/
            /blue/shiboli.fsu/yifeisun.umich/adversarial_robustness_FNO/1D_Burgers_FNO_generalization/datasets/all_neg/gaussian_diff_distr/gaussiandiffdistr_cl0.005_dim=1d_solver=exponax_kernel=gaussian_correlation_length=0.005_bc=periodic_nu=0.0005_t=1.0_seed=3_N=5.pkl
            /blue/shiboli.fsu/yifeisun.umich/adversarial_robustness_FNO/1D_Burgers_FNO_generalization/datasets/all_neg/gaussian_diff_distr/gaussiandiffdistr_cl0.005_dim=1d_solver=exponax_kernel=gaussian_correlation_length=0.005_bc=periodic_nu=0.0005_t=1.0_seed=3_N=5.pt
            /blue/shiboli.fsu/yifeisun.umich/adversarial_robustness_FNO/1D_Burgers_FNO_generalization/datasets/

In [ ]:

# expanded models 用某些attack之后得到的结果作为数据集的扩充训练的模型  
root = "../1D_Burgers/saved_models_expanded"
print_tree(root)

"../1D_Burgers/saved_models/1D/modes16_width64_epochs500/pos/old/unnormalized/burgers_1d_FNO_model_trainedby_dim1d_nx1024_N1500_solver=exponax_kernel=gaussian_correlation_length0.03_bcperiodic_nu0.0005_t1.0_seed45.pth"

/blue/shiboli.fsu/yifeisun.umich/adversarial_robustness_FNO/1D_Burgers/saved_models_expanded/
    /blue/shiboli.fsu/yifeisun.umich/adversarial_robustness_FNO/1D_Burgers/saved_models_expanded/1D/
        /blue/shiboli.fsu/yifeisun.umich/adversarial_robustness_FNO/1D_Burgers/saved_models_expanded/1D/modes16_width64_epochs500/
            /blue/shiboli.fsu/yifeisun.umich/adversarial_robustness_FNO/1D_Burgers/saved_models_expanded/1D/modes16_width64_epochs500/unnormalized/
                /blue/shiboli.fsu/yifeisun.umich/adversarial_robustness_FNO/1D_Burgers/saved_models_expanded/1D/modes16_width64_epochs500/unnormalized/add100/
                    /blue/shiboli.fsu/yifeisun.umich/adversarial_robustness_FNO/1D_Burgers/saved_models_expanded/1D/modes16_width64_epochs500/unnormalized/add100/norm2_eps20_alpha0.1_steps20/
                        /blue/shiboli.fsu/yifeisun.umich/adversarial_robustness_FNO/1D_Burgers/saved_models_expanded/1D/modes16_width64_epochs500/unnormalized/add100/norm2_eps

In [ ]:
import os
import pickle
import torch

def convert_pkl_to_pt(pkl_path):
    pt_path = pkl_path.replace(".pkl", ".pt")
    with open(pkl_path, "rb") as f:
        records = pickle.load(f)

    x_list, y_list = [], []
    for rec in records:
        # "a" 是输入 (numpy)，"g(a)" 是输出 (tensor)
        a = torch.as_tensor(rec["a"], dtype=torch.float32)
        g = torch.as_tensor(rec["g(a)"], dtype=torch.float32)
        x_list.append(a)
        y_list.append(g)

    x = torch.stack(x_list, dim=0)
    y = torch.stack(y_list, dim=0)

    pt_obj = {"x": x, "y": y, "t_final": 1.0}
    torch.save(pt_obj, pt_path)

    print(f"[OK] {pkl_path} → {pt_path}, x={x.shape}, y={y.shape}")

def batch_convert(root="datasets"):
    for dirpath, _, filenames in os.walk(root):
        for fname in filenames:
            if fname.endswith(".pkl"):
                pkl_path = os.path.join(dirpath, fname)
                convert_pkl_to_pt(pkl_path)

if __name__ == "__main__":
    batch_convert("datasets")



In [3]:
# -*- coding: utf-8 -*-
import os
import torch
import numpy as np
from pathlib import Path

# =========================
# 配置
# =========================
ROOTS = [
    "datasets",
    "../1D_Burgers/datasets/1D/Burgers/expanded_pos/t1",
    "test_train_datasets",
]
ALLOWED_EXTS = {".pt", ".pth"}

# PyTorch 2.6+ 默认 weights_only=True；这里明确用开关控制。
# ⚠️ 仅当你信任这些 .pt/.pth 文件来源时，才建议将此值设为 False（完全反序列化）。
WEIGHTS_ONLY = False


# =========================
# 工具函数
# =========================
def load_any(path, map_location="cpu", weights_only=False):
    """
    统一加载入口：
    - weights_only=False：完全反序列化（可能执行任意代码，仅在信任来源时使用）。
    - weights_only=True ：安全模式；如果遇到 numpy 标量类型不在 allowlist 的错误，
                          先将其加入 allowlist 再尝试一次。
    """
    if not weights_only:
        return torch.load(path, map_location=map_location, weights_only=False)

    try:
        return torch.load(path, map_location=map_location, weights_only=True)
    except Exception:
        # 遇到报错里建议允许的 numpy 标量类型，按需加入 allowlist 再试一次
        try:
            torch.serialization.add_safe_globals([np.core.multiarray.scalar])
        except Exception:
            # 有些环境可能没有该属性，忽略并继续下一步尝试
            pass
        return torch.load(path, map_location=map_location, weights_only=True)


def is_tensor_like(x):
    return isinstance(x, (torch.Tensor, np.ndarray))


def shape_of(x):
    if isinstance(x, torch.Tensor):
        return tuple(x.shape)
    if isinstance(x, np.ndarray):
        return x.shape
    return None


def short_repr(x, max_len=120):
    s = repr(x)
    if len(s) > max_len:
        s = s[:max_len - 3] + "..."
    return s


def describe_loaded_object(obj, indent="    "):
    """
    将顶层对象（常见是 dict）格式化为可读文本：
    - dict：打印 key、类型、shape/长度/值简写
    - 张量/ndarray：打印 dtype、device、shape
    - list/tuple：打印前 3 个元素的类型与 shape
    - 其他：打印类型与值简写
    """
    lines = []
    if isinstance(obj, dict):
        lines.append(f"{indent}dict with {len(obj)} keys")
        for k, v in obj.items():
            shp = shape_of(v)
            base = f"{indent}- {k!r}: {type(v).__name__}"
            if shp is not None:
                base += f", shape={shp}"
            elif isinstance(v, (list, tuple)):
                base += f", len={len(v)}"
            elif isinstance(v, dict):
                base += f", subkeys={len(v)}"
            else:
                base += f", value={short_repr(v)}"
            lines.append(base)

    elif is_tensor_like(obj):
        shp = shape_of(obj)
        dtype = str(obj.dtype) if hasattr(obj, "dtype") else ""
        device = str(obj.device) if hasattr(obj, "device") else ""
        lines.append(f"{indent}{type(obj).__name__}, dtype={dtype}, device={device}, shape={shp}")

    elif isinstance(obj, (list, tuple)):
        lines.append(f"{indent}{type(obj).__name__}, len={len(obj)}")
        for i, v in enumerate(obj[:3]):
            shp = shape_of(v)
            line = f"{indent}  [{i}] {type(v).__name__}"
            if shp is not None:
                line += f", shape={shp}"
            else:
                line += f", value={short_repr(v)}"
            lines.append(line)
        if len(obj) > 3:
            lines.append(f"{indent}  ... ({len(obj)-3} more)")

    else:
        lines.append(f"{indent}{type(obj).__name__}, value={short_repr(obj)}")

    return "\n".join(lines)


def find_dataset_files(roots, exts=ALLOWED_EXTS):
    files = []
    for root in roots:
        p = Path(root)
        if not p.exists():
            print(f"[WARN] Not found: {p.resolve()}")
            continue
        for f in p.rglob("*"):
            if f.is_file() and f.suffix.lower() in exts:
                files.append(f.resolve())
    files.sort()
    return files


def inspect_datasets():
    files = find_dataset_files(ROOTS)
    print(f"[INFO] Found {len(files)} dataset file(s).\n")

    for i, f in enumerate(files, 1):
        print("=" * 80)
        print(f"[{i}/{len(files)}] File: {str(f)}")
        try:
            obj = load_any(f, map_location="cpu", weights_only=WEIGHTS_ONLY)
        except Exception as e:
            print(f"[ERROR] Failed to load: {e}")
            continue

        try:
            desc = describe_loaded_object(obj, indent="    ")
            print(desc)
        except Exception as e:
            print(f"[ERROR] Failed to describe object: {e}")
            print(f"    Top-level type fallback: {type(obj).__name__}")


# =========================
# 运行
# =========================
if __name__ == "__main__":
    inspect_datasets()



[INFO] Found 206 dataset file(s).

[1/206] File: /blue/shiboli.fsu/yifeisun.umich/adversarial_robustness_FNO/1D_Burgers/datasets/1D/Burgers/expanded_pos/t1/nu0.0005_pgd/norm2_eps10_alpha0.01_steps20/N=1500/dataset.pt
    dict with 3 keys
    - 'x': Tensor, shape=(1500, 1024)
    - 'y': Tensor, shape=(1500, 1024)
    - 't_final': float, value=1.0
[2/206] File: /blue/shiboli.fsu/yifeisun.umich/adversarial_robustness_FNO/1D_Burgers/datasets/1D/Burgers/expanded_pos/t1/nu0.0005_pgd/norm2_eps10_alpha0.05_steps20/N=1500/dataset.pt
    dict with 3 keys
    - 'x': Tensor, shape=(1500, 1024)
    - 'y': Tensor, shape=(1500, 1024)
    - 't_final': float, value=1.0
[3/206] File: /blue/shiboli.fsu/yifeisun.umich/adversarial_robustness_FNO/1D_Burgers/datasets/1D/Burgers/expanded_pos/t1/nu0.0005_pgd/norm2_eps10_alpha0.1_steps20/N=1500/dataset.pt
    dict with 3 keys
    - 'x': Tensor, shape=(1500, 1024)
    - 'y': Tensor, shape=(1500, 1024)
    - 't_final': float, value=1.0
[4/206] File: /blue/shiboli